In [ ]:
pip install transformers scikit-learn torch numpy pandas peft ollama numpy

In [ ]:
#choose model
model_name = "llama3.1:8b-instruct-q8_0"

#pull mode
!ollama pull {model_name}

!ollama list

pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest ⠧ pulling manifest ⠇ pulling manifest 
pulling 667b0c1932bc: 100% ▕██████████████████▏ 4.9 GB                         
pulling 948af2743fc7: 100% ▕██████████████████▏ 1.5 KB                         
pulling 0ba8f0e314b4: 100% ▕██████████████████▏  12 KB                         
pulling 56bb8bd477a5: 100% ▕██████████████████▏   96 B                         
pulling 455f34728c9b: 100% ▕██████████████████▏  487 B                         
verifying sha256 digest 
writing manifest 
success 
NAME                                                  ID              SIZE      MODIFIED               
llama3.1:8b                                           46e0c10c039e    4.9 GB    Less than a second ago    
llama3.1:8b-instruct-q8_0                             b158ded76fa0    8.5 GB    23 hours ago              
llama3.1:8b-instruct-q8_0_framing_ft_fram

In [2]:
#import packages and datasets

import pandas as pd
import numpy as np
import sklearn as sk
import json
import ollama
from pathlib import Path

#unlabelled dataset
df = pd.read_csv("../unlabelled_relevance.csv")

#labelled_dataset from Annotator 1 in round 2
df_gold = pd.read_csv('../Human/A1_r2.csv')

with open('../relevance_codebook.json', 'r') as f:
    codebook = json.load(f)
    
codebook_str = json.dumps(codebook, indent=2, ensure_ascii=False)


In [ ]:
#helper functions

##parsing responses from model
def parse_json_with_fallback(content_str):
    # Strip whitespace
    content_str = content_str.strip()
    # If the string is supposed to end with '}', but doesn't, add it.
    if not content_str.endswith('}'):
        content_str += '}'
    # Now try parsing
    try:
        dict_val = json.loads(content_str)
        str_val = dict_val.get('Relevance')
        binary_val = 1 if str_val.strip().lower() == "yes" else 0

        return binary_val
    except json.JSONDecodeError:
        return pd.NA



##compute kappa and accuracy
def compute_scores(df_pred, df_gold, column_pred, column_gold, column_match):
    df_pred_reduced = df_pred[[column_match, column_pred]]
    df_gold_reduced = df_gold[[column_match, column_gold]]

    #join these two dataframes
    df_temp = (
        df_pred_reduced
            .merge(df_gold_reduced, on=column_match, how='inner')   # keep only matching IDs
            .dropna(subset=[column_pred, column_gold])          # drop rows where values are NaN
            .reset_index(drop=True)                                 # tidy up the index
    )
    
    #ensure that the columns have data in the same type 
    df_temp[column_pred] = df_temp[column_pred].astype(int)
    df_temp[column_gold] = df_temp[column_gold].astype(int)

    #compute scores
    acc = round(100*sk.metrics.accuracy_score(df_temp[column_gold], df_temp[column_pred]), 3)
    k = round(sk.metrics.cohen_kappa_score(df_temp[column_gold], df_temp[column_pred]), 3)

    print("Accuracy is " + str(acc) + "%, and Cohen's kappa is " + str(k))
    
#formatting fine tuning dataset to JSONL and saving
def to_ollama_jsonl(df_in: pd.DataFrame,
                    out_path: str | Path,
                    system_prompt: str,
                    user_prompt: str) -> Path:
    """
    Convert <article text, label> rows into Ollama‑ready JSONL.
    Each line is one conversation:  system ➜ user ➜ assistant
    """
    out_path = Path(out_path)
    with out_path.open("w", encoding="utf‑8") as f:
        for _, row in df_in.iterrows():
            label = row["gold"]
            assistant = "yes" if label == 1 else "no"

            record = {
                "messages": [
                    {"role": "system", "content": system_prompt},
                    {"role": "user",   "content": user_prompt + row["text"]},
                    {"role": "assistant", "content": assistant}
                ]
            }
            json.dump(record, f, ensure_ascii=False)
            f.write("\n")
    return out_path



In [ ]:
#Prompts
SYSTEM_PROMPT = """You're a communication researcher who is studying the news reporting of Mpox. You’ll perform relevance coding on news articles, by classifying articles as relevant, or not relevant. 
An article is marked relevant if it contains explicit content directly discussing Mpox in relation to the human epidemic context. 
An article is marked Irrelevant if it does not substantively discuss Mpox in a way related to the epidemic, or mentions it only in passing, satirical, or non-human contexts.
Remember to prioritize accuracy and clarity in your analysis, using the provided context and your expertise to guide your evaluation."""


USER_PROMPT =  """Is the following article relevant?  Provide your response in a JSON array format, as follows, and include nothing else in the response : { "Relevance": "yes/no" }. 
If you are uncertain about the classification, force a decision to choose "yes" or "no". """

In [ ]:
#Zero Shot relevance binary classification - no codebook

#sample for testing things first
#df_sample = df.sample(n= 30, random_state= 42).reset_index()

#otherwise
df_sample = df

df1 = df_sample

predictions = list()

for i in range(len(df1)):
    try:
        messages = [
        {"role": "system", 
         "content": SYSTEM_PROMPT
        },

        {
         "role": "user", 
         "content": USER_PROMPT + df1["text"][i] 
        }

            ]
    
        outputs = ollama.chat(model= model_name, messages= messages)

        predictions.append(outputs.message.content)
    
    #if there is an error
    except Exception as e:
            predictions.append(None)

    
    if(i%10 == 0): print(str(i) + " iterations finished")


#save the responses
RelevanceList = []

for output in predictions:
    content_str = output
    
    annotation = parse_json_with_fallback(content_str)

    RelevanceList.append(annotation)

df1["relevance"] = RelevanceList

#compute metrics
compute_scores(df_gold= df_gold, df_pred = df1, column_match= 'stories_id', column_gold= 'gold', column_pred= 'relevance')

df1.to_csv("Predicted/" + model_name + "_zshot_noCB.csv")

In [ ]:
#Zero Shot relevance binary classification - with codebook

#sample for testing things first
#df_test = df.sample(n= 30, random_state= 42).reset_index()
df_test = df

df1 = df_test

predictions = list()

for i in range(len(df1)):
    try:
        messages = [
        {"role": "system", 
         "content": SYSTEM_PROMPT + codebook_str
        },

        {
         "role": "user", 
         "content": USER_PROMPT + df1["text"][i] 
        }

            ]
    
        outputs = ollama.chat(model= model_name, messages= messages)

        predictions.append(outputs.message.content)
    
    #if there is an error
    except Exception as e:
            predictions.append(None)

    
    if(i%10 == 0): print(str(i) + " iterations finished")

#save the responses
RelevanceList = []

for output in predictions:
    content_str = output
    
    annotation = parse_json_with_fallback(content_str)

    RelevanceList.append(annotation)

df1["relevance"] = RelevanceList

#compute metrics
compute_scores(df_gold= df_gold, df_pred = df1, column_match= 'stories_id', column_gold= 'gold', column_pred= 'relevance')

df1.to_csv("Predicted/" + model_name + "_zshot_withCB.csv")

In [ ]:
#Fine tuning with 60% of the annotated data

#sample some for testing the code first
#df_sample = df_gold.sample(n= 30, random_state= 42).reset_index()
df_sample = df_gold

#split df into tune and test (same train-test split as BERT)
df_tune, df_test = sk.model_selection.train_test_split(df_sample, test_size= 0.4, random_state=5)

#reset indices
df_tune = df_tune.reset_index()
df_test = df_test.reset_index()

#format the fine tuning data
tune_file = to_ollama_jsonl(df_tune, "relevance_tune.jsonl", SYSTEM_PROMPT, USER_PROMPT)

#Let the fine tuning begin
ft_model_name = model_name + "relevance_ft"

client = ollama.Client()                       # defaults to http://localhost:11434
digest = client.create_blob(tune_file)         # uploads the JSONL, returns sha256
progress = client.create(
    model       = ft_model_name,     # new tag
    from_       = model_name,                  # parent model
    files       = {"relevance_tune.jsonl": digest},
    parameters  = {"num_epochs": 3},           # any ggml‑compatible training args
    stream      = True                         # chunked progress
)
for chunk in progress:                         # stream shows download / training bar
    print(chunk.status, chunk.completed, "/", chunk.total)

#annotate test set with fine tuned model
df1 = df_test

predictions = list()

for i in range(len(df1)):
    try:
        messages = [
        {"role": "system", 
         "content": SYSTEM_PROMPT
        },

        {
         "role": "user", 
         "content": USER_PROMPT + df1["text"][i] 
        }

            ]
    
        outputs = ollama.chat(model= ft_model_name, messages= messages)

        predictions.append(outputs.message.content)
    
    #if there is an error
    except Exception as e:
            predictions.append(None)

    
    if(i%10 == 0): print(str(i) + " iterations finished")


#save the responses
RelevanceList = []

for output in predictions:
    content_str = output
    
    annotation = parse_json_with_fallback(content_str)

    RelevanceList.append(annotation)

df1["relevance"] = RelevanceList

#compute metrics
compute_scores(df_gold= df_gold, df_pred = df1, column_match= 'stories_id', column_gold= 'gold', column_pred= 'relevance')

df1.to_csv("Predicted/" + ft_model_name + ".csv")